# Phase 3 - Step 3: experiment #1 = A2 (decoder + copy) **full fine-tuning**, context-aware

The first real model. It is also the **teacher** for the knowledge-distillation experiment (#4).

**Model** (`src/rr_models.py`):

1. The document is cut into windows of whole sentences (<= 1536 tokens, ~36 sentences each).
2. The A2 backbone (Phase-1 weights, causal = left-to-right reading as in pretraining) reads each window.
3. Each sentence = the average of its token vectors, taken from the window where it is most central.
4. To every sentence vector we add its relative position (0 = first sentence, 1 = last) and a small
   learned CL/IT vector.
5. A BiLSTM reads ALL sentences of the document (left->right and right->left), then a Linear layer
   gives 13 role scores per sentence.

**Full fine-tuning** = every weight is trained (backbone 70M + head ~1.6M).
Loss = cross-entropy with square-root class weights. One step = 2 documents.
Backbone lr 1e-4, head lr 1e-3 (the head starts from random), warm-up 10%, linear decay, up to 15 epochs,
early stop after 4 epochs without improvement of dev macro-F1.

**Stage A**: train on CL+IT train (80 docs), check dev (10 docs) after every epoch -> best epoch (reported numbers).
**Stage B**: fresh Phase-1 weights, train on train+dev (90 docs) for the best epoch count -> test submission.

Everything is cached in `checkpoints/phase3/A2_full_causal_doc/`: re-running a cell loads finished results
instead of training again, and an interrupted training resumes from the last finished epoch.


In [1]:
import sys, json, time
from pathlib import Path
import numpy as np
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, rr_utils, rr_models, rr_experiment
for m in (rr_utils, rr_models, rr_experiment):
    importlib.reload(m)
from rr_experiment import Experiment, compare
import rr_utils as ru, rr_models as rm

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU would be far too slow)")

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## 1. Build the experiment and check memory on the longest document

Before a 30-minute run we do ONE training step on the longest training document (71k tokens, 47 windows,
1,816 sentences): if this fits in 6 GB, every document fits. The step does not change the saved model.


In [2]:
exp = Experiment(ROOT, {"phase1": "A2", "method": "full"})
print(json.dumps({k: v for k, v in exp.cfg.items() if k not in ("adapter_targets",)}, indent=None))

longest = max(exp.enc["train"], key=lambda d: len(d.ids))
wins, owner = exp.plans[longest.id]
print(f"longest training document: {len(longest.ids):,} tokens, {longest.n_sent:,} sentences, {len(wins)} windows")

check_path = exp.dir / "memory_check.json"
if check_path.exists():
    print("[cache] memory check:", json.load(open(check_path)))
else:
    model = exp.build_model()
    total, trainable = rm.count_params(model)
    print(f"parameters: {total / 1e6:.2f}M total, {trainable / 1e6:.2f}M trainable | {exp.load_info}")
    model.train()
    if exp.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    logits = model.forward_doc(longest, wins, owner, exp.device, exp.amp, exp.cfg["win_batch"], use_ckpt=True)
    loss = torch.nn.functional.cross_entropy(logits, torch.from_numpy(longest.y).to(exp.device))
    loss.backward()
    if exp.device.type == "cuda":
        torch.cuda.synchronize()
    res = {"tokens": int(len(longest.ids)), "sec": round(time.time() - t0, 1), "loss_at_start": round(loss.item(), 3),
           "peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if exp.device.type == "cuda" else None}
    print("forward + backward on the longest document:", res, "(loss ~ ln 13 = 2.56 for an untrained head)")
    json.dump(res, open(check_path, "w"))
    del model, logits, loss
    torch.cuda.empty_cache()

experiment: A2_full_causal_doc | device cuda | precision torch.bfloat16
{"phase1": "A2", "method": "full", "causal": true, "context": "doc", "max_len": 1536, "stride": 1024, "max_sent_tokens": 256, "head": "bilstm", "hidden": 256, "dropout": 0.2, "use_pos": true, "use_domain": true, "class_weight_power": 0.5, "epochs": 15, "patience": 4, "docs_per_step": 2, "win_batch": 2, "lr": 0.0001, "head_lr": 0.001, "weight_decay": 0.01, "warmup_frac": 0.1, "grad_clip": 1.0, "seed": 42, "rank": 8, "alpha": 16, "kd_teacher": null, "kd_alpha": 0.5, "kd_T": 2.0, "phase1_run": "A2_copy1_mtp0_d640_L10_sc70tr30_200M"}
longest training document: 70,777 tokens, 1,809 sentences, 67 windows
parameters: 71.94M total, 71.94M trainable | {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}
forward + backward on the longest document: {'tokens': 70777, 'sec': 7.5, 'loss_at_start': 2.55, 'peak_gb': 1.87} (loss ~ ln 13 = 2.56 for a

## 2. Stage A: train on train, validate on dev every epoch (~1-2 min per epoch)


In [3]:
state_a = exp.stage_a()

parameters: 71.94M total, 71.936M trainable | loaded {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}


epoch  1 | train loss 2.0378 | 45s | dev macro-F1 0.3864 (CL 0.3310, IT 0.3732) acc 0.5107  <- best


epoch  2 | train loss 1.2461 | 45s | dev macro-F1 0.5025 (CL 0.4368, IT 0.4927) acc 0.5486  <- best


epoch  3 | train loss 0.9204 | 47s | dev macro-F1 0.4857 (CL 0.4545, IT 0.4588) acc 0.5373


epoch  4 | train loss 0.6874 | 46s | dev macro-F1 0.5235 (CL 0.4926, IT 0.5191) acc 0.5570  <- best


epoch  5 | train loss 0.4795 | 47s | dev macro-F1 0.4849 (CL 0.4140, IT 0.5317) acc 0.5789


... (2 outputs left out)

epoch  8 | train loss 0.1568 | 46s | dev macro-F1 0.5515 (CL 0.4703, IT 0.5760) acc 0.6027


epoch  9 | train loss 0.0978 | 46s | dev macro-F1 0.4911 (CL 0.4020, IT 0.5626) acc 0.5999


epoch 10 | train loss 0.0633 | 46s | dev macro-F1 0.5100 (CL 0.4435, IT 0.5501) acc 0.6204


epoch 11 | train loss 0.0340 | 47s | dev macro-F1 0.5101 (CL 0.4391, IT 0.5614) acc 0.6176
early stop: no improvement for 4 epochs
training 'train' finished -> loaded best epoch 7
Stage A: best epoch 7 | dev macro-F1 0.5616 | 538s | peak GPU 2.75 GB


## 3. Dev report

Macro-F1 for CL+IT together and per domain, F1 per role, the most frequent confusions, and the two baselines
(majority 0.043, position only 0.169) for comparison.


In [4]:
dev_scores = exp.report()

A2_full_causal_doc - dev:
  all: macro-F1 0.5616 | macro-P 0.5844 | macro-R 0.5712 | accuracy 0.6228 | macro-F1 over all 13 0.4752 | 11 labels, 2142 sentences
  CL : macro-F1 0.4883 | macro-P 0.4951 | macro-R 0.5064 | accuracy 0.6447 | macro-F1 over all 13 0.4132 | 11 labels, 1396 sentences
  IT : macro-F1 0.5434 | macro-P 0.5972 | macro-R 0.5329 | accuracy 0.5818 | macro-F1 over all 13 0.4598 | 11 labels, 746 sentences
  label                      all F1    CL F1    IT F1   all n    CL n    IT n
  Fact                        0.715    0.686    0.755     444     249     195
  Issue                       0.629    0.250    0.741      15       3      12
  ArgumentPetitioner          0.574    0.575    0.571     400     344      56
  ArgumentRespondent          0.561    0.575    0.464     272     236      36
  Statute                     0.700    0.621    0.793     120      58      62
  Dissent                     0.000    0.000    0.000       0       0       0
  PrecedentReliedUpon         

## 4. Stage B: retrain on train + dev, predict the test documents, write the submission

`checkpoints/phase3/A2_full_causal_doc/submission/submission.zip` -> upload to Codabench (Phase 3 submission 1).


In [5]:
zip_path = exp.stage_b()
print(zip_path)

epoch  1 | train loss 1.9067 | 52s


epoch  2 | train loss 1.1522 | 52s


epoch  3 | train loss 0.8219 | 51s


epoch  4 | train loss 0.5659 | 52s


epoch  5 | train loss 0.4240 | 52s


epoch  6 | train loss 0.2582 | 50s


epoch  7 | train loss 0.1561 | 50s
training 'full' finished -> loaded best epoch 7


submission OK: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase3\A2_full_causal_doc\submission\submission.zip | 10 documents | 2,197 sentences | label counts {'RatioOfTheDecision': 790, 'Fact': 422, 'PrecedentReliedUpon': 267, 'ArgumentPetitioner': 185, 'Statute': 167, 'ArgumentRespondent': 141, 'RulingByLowerCourt': 107, 'RulingByPresentCourt': 49, 'None': 26, 'PrecedentNotReliedUpon': 19, 'Issue': 17, 'Dissent': 7}
C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase3\A2_full_causal_doc\submission\submission.zip


## 5. Efficiency + log


In [6]:
exp.speed()
exp.log()
compare(ROOT)

speed: {'tokens_per_sec': 98605, 'sentences_per_sec': 1814, 'infer_sec_dev': 1.18, 'peak_gb_infer': 0.47, 'params_total': 71935661, 'params_trainable': 71935661}
logged: A2_full_causal_doc | dev macro-F1 0.5616
run                                     macro     CL     IT    acc  ep     Fact   Issue Argumen Argumen Statute Dissent Precede Precede Precede RulingB RatioOf RulingB    None
A2_full_causal_doc                     0.5616 0.4883 0.5434 0.6228   7    0.715   0.629   0.574   0.561   0.700   0.000   0.530   0.371   0.000   0.529   0.713   0.565   0.292


[('A2_full_causal_doc',
  0.5615739186340778,
  0.4883305964462039,
  0.5433891559240367,
  0.6227824463118581,
  7,
  {'Fact': 0.7148,
   'Issue': 0.6286,
   'ArgumentPetitioner': 0.5742,
   'ArgumentRespondent': 0.5612,
   'Statute': 0.6996,
   'Dissent': 0.0,
   'PrecedentReliedUpon': 0.53,
   'PrecedentNotReliedUpon': 0.3709,
   'PrecedentOverruled': 0.0,
   'RulingByLowerCourt': 0.5287,
   'RatioOfTheDecision': 0.7125,
   'RulingByPresentCourt': 0.5652,
   'None': 0.2917})]